# Разработка чат-бота с трансформерами
Задание:  
1. Загрузить предобученную модель (Hugging Face).  
2. Дообучить на датасете диалогов (например, Cornell Movie Dialogs).  
3. Реализовать инференс и генерацию ответов.  

Порядок выполнения:
1. Подготовка данных (токенизация, разбивка на реплики).  
2. Fine-tuning с AdamW.
3. Генерация ответов на тестовые фразы.

In [1]:
!pip install transformers datasets torch sentencepiece accelerate

In [2]:
import os
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
!wget -q https://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip
!unzip -q cornell_movie_dialogs_corpus.zip

In [91]:
path = 'cornell movie-dialogs corpus'

with open(f'{path}/movie_lines.txt', 'r', encoding='iso-8859-1') as f:
    lines = f.readlines()

line_id_to_text = {}
for line in lines:
    parts = line.split(' +++$+++ ')
    line_id_to_text[parts[0]] = parts[-1].strip()

In [92]:
with open(f'{path}/movie_conversations.txt', 'r', encoding='iso-8859-1') as f:
    conversations = f.readlines()

qa_pairs = []

for convo in conversations[:2000]:
    parts = convo.split(' +++$+++ ')
    line_ids = eval(parts[-1])  # строка вида ['L123', 'L456', ...] → список

    for i in range(len(line_ids) - 1):
        q_id = line_ids[i]
        a_id = line_ids[i + 1]
        if q_id in line_id_to_text and a_id in line_id_to_text:
            question = line_id_to_text[q_id]
            answer = line_id_to_text[a_id]
            qa_pairs.append({
                'input': question,
                'target': answer
            })

df = pd.DataFrame(qa_pairs)
print(f"Всего пар реплик: {len(df)}")
print("\nПримеры:")
print(df.head(5))

Всего пар реплик: 4863

Примеры:
                                               input  \
0  Can we make this quick?  Roxanne Korrine and A...   
1  Well, I thought we'd start with pronunciation,...   
2  Not the hacking and gagging and spitting part....   
3  You're asking me out.  That's so cute. What's ...   
4  No, no, it's my fault -- we didn't have a prop...   

                                              target  
0  Well, I thought we'd start with pronunciation,...  
1  Not the hacking and gagging and spitting part....  
2  Okay... then how 'bout we try out some French ...  
3                                         Forget it.  
4                                           Cameron.  


In [93]:
# Используем GPT-2 токенизатор
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 не имеет pad_token

In [94]:
def format_conversation(ex):
    return f"Question: {ex['input']} Answer: {ex['target']}<|endoftext|>"

# Применяем к датасету
df['text'] = df.apply(format_conversation, axis=1)

# Преобразуем в Hugging Face Dataset
dataset = Dataset.from_pandas(df[['text']])

In [95]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors=None  # возвращаем списки
    )

# Применяем
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=['text'])

# Разделение на train/test
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)
print("Размер обучающей выборки:", len(split_dataset['train']))
print("Размер тестовой:", len(split_dataset['test']))

Map:   0%|          | 0/4863 [00:00<?, ? examples/s]

Размер обучающей выборки: 4376
Размер тестовой: 487


In [96]:
# Загрузка модели
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Data collator для языкового моделирования
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # для causal LM (GPT)
)

In [97]:
training_args = TrainingArguments(
    output_dir='./gpt2-dialogue',
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1000,
    logging_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    warmup_steps=200,
    lr_scheduler_type='linear',
    fp16=True,  # если GPU поддерживает
    logging_dir='./logs',
    report_to='none',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False
)

In [98]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset['train'],
    eval_dataset=split_dataset['test'],
    data_collator=data_collator,
    tokenizer=tokenizer
)

/tmp/ipython-input-1924320405.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [99]:
print("Запуск дообучения...")
trainer.train()

# Сохранение модели
model.save_pretrained('./gpt2-dialogue-finetuned')
tokenizer.save_pretrained('./gpt2-dialogue-finetuned')
print("Модель сохранена!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Запуск дообучения...


Step,Training Loss,Validation Loss


Модель сохранена!


In [100]:
def generate_response(question, model, tokenizer, max_length=100, ):
    input_text = f"Question: {question} Answer:"
    inputs = tokenizer(input_text, return_tensors='pt', truncation=True, max_length=100).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=0.9,
            top_k=50,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            num_return_sequences=1
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Извлекаем только часть после "Ответ:"
    if "Ответ:" in response:
        answer = response.split("Ответ:")[1].strip()
    else:
        answer = response[len(input_text):].strip()

    return answer

In [101]:
# Загрузка дообученной модели (если перезапускаем)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Примеры
questions = [
    # "Hi, how are you doing?",
    # "Who are you?",
    # "Are you a robot?",
    "What is the capital of Russia?",
]

print("🤖 ДЕМО-ДИАЛОГ С ДООБУЧЕННЫМ GPT-2\n")
for q in questions:
    response = generate_response(q, model, tokenizer)
    print(f"👤 Вы: {q}")
    print(f"🤖 Бот: {response}")
    print("-" * 50)

🤖 ДЕМО-ДИАЛОГ С ДООБУЧЕННЫМ GPT-2

👤 Вы: What is the capital of Russia?
🤖 Бот: Nothing. It's just a stupid old thing, you know. All you have to do is send a few hundred thousand dollars and it'll get us the job.  But we don't know where you got it. Answer: That's the way it is.  I think you can find it.  Here it is, in a nice little room.  The men are busy waiting for you to get in.  What the hell are
--------------------------------------------------


Задание 1

In [102]:
def generate_response_difTemp(question, model, tokenizer, max_length=100, temp = 0.9):
    input_text = f"Question: {question} Answer:"
    inputs = tokenizer(input_text, return_tensors='pt', truncation=True, max_length=100).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=temp,
            top_k=50,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            num_return_sequences=1
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Извлекаем только часть после "Ответ:"
    if "Ответ:" in response:
        answer = response.split("Ответ:")[1].strip()
    else:
        answer = response[len(input_text):].strip()

    return answer

In [103]:
# Загрузка дообученной модели (если перезапускаем)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Примеры
questions = [
    "Hi, how are you doing?",
    # "Who are you?",
    "Are you a robot?",
    #"What is the capital of Russia?",
]

print("🤖 ДЕМО-ДИАЛОГ С ДООБУЧЕННЫМ GPT-2\n")
for q in questions:
    print("temperature = 0.3")
    response = generate_response_difTemp(q, model, tokenizer, temp = 0.3)
    print(f"👤 Вы: {q}")
    print(f"🤖 Бот: {response}")
    print("-" * 50)
    print("temperature = 0.7")
    response = generate_response_difTemp(q, model, tokenizer, temp = 0.7)
    print(f"👤 Вы: {q}")
    print(f"🤖 Бот: {response}")
    print("-" * 50)
    print("temperature = 1.2")
    response = generate_response_difTemp(q, model, tokenizer, temp = 1.2)
    print(f"👤 Вы: {q}")
    print(f"🤖 Бот: {response}")
    print("-" * 50)

🤖 ДЕМО-ДИАЛОГ С ДООБУЧЕННЫМ GPT-2

temperature = 0.3
👤 Вы: Hi, how are you doing?
🤖 Бот: Well, I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm fine.  I'm
--------------------------------------------------
temperature = 0.7
👤 Вы: Hi, how are you doing?
🤖 Бот: Good. I'm fine. I'll go home and find some food.  I'll be back soon.  It's going well.  Good luck. I'm glad you're back.  I know you've been so busy.  I'm sure you're still very busy.  It's good to see you again.  I hope you have a good weekend. I think you're very well. Goodnight.  Have a
--------------------------------------------------
temperature = 1.2
👤 Вы: Hi, how are you doing?
🤖 Бот: Nothing...except the heat. It's getting really hot from everywhere! The room is hot! It's not worth waiting, okay?  What, the windows are open on the back?  Where?  Where? Oh, I don't know, there's nothin' at my d